# R16c step 4 · SAM 3 on the training images, for a grader trained on SAM 3's own outlines

Task **R16c**, step 4, from `Essential_task_delegation/Progress_Tracker.md`. It was called step 3a until R16c's steps were renumbered 1–5 on 20 Oct 2026. Step 2 was kept, so by its own rule this step wasn't needed; you chose on 20 Oct to run it as a stretch.

Built as a copy of `e3-sam3-building-finding.ipynb`, which stays unchanged. It keeps E3's SAM 3 commit, its two T4 fixes, its prompt and confidence, and fp16 (E3's choice). It runs only E3's best run: the before-image as a 2×2 grid (`pre_grid2`, building IoU 0.630 on the validation images).

**Why**
- R16c's grader learns on the true building pieces, but it is judged on SAM 3's outlines. On the 645 validation images, a perfect grader on those outlines would reach Destroyed IoU 0.614.
- This notebook makes SAM 3's outlines of the 5,798 training images. Step 5 then trains the grader on the outlines it will grade: `r16c-step5-grader-on-sam3-outlines.ipynb`.

**What it does**
- SAM 3, prompt "building", on R16's 5,798 training before-images, as a 2×2 grid. Each image gets one 16-bit instance map.
- Per image it also records building IoU against the true mask, the instance count and the footprint recall per class, as E3 did, so these outlines can be compared with E3's validation numbers.
- **Resume:** maps already written are skipped. Attach a stopped version's output and run again to continue.

**How to run on Kaggle** (account 1, whose Hugging Face token has SAM 3 access)
1. Settings: GPU T4 ×1, internet ON, secret `HF_TOKEN` (AbrarAlam, approved for `facebook/sam3`).
2. Input: the DisasterM3 mirror (attached automatically).
3. Run All once with `SMOKE_TEST = True` (8 images, a few minutes after the install). Then set it to `False` and use Save & Run All: about 3.5–4 hours (E3 took 2.1 s per image).
4. Outputs in `/kaggle/working/step4/`:
   - `masks/pre_grid2_train/<after-image name>.png`: 16-bit, 0 = background. Step 5 finds this folder by its name.
   - `sam3_train.csv`, `summary.csv`, `config.json`, `README.md`.

In [1]:
# ── Cell 1: Install SAM 3 (pinned commit) and log in to Hugging Face ────────
# SAM 3 pins numpy<2, which has no wheel for Kaggle's Python 3.13, so it is installed without its
# dependency list. These are the packages it really imports at load time (checked against the source).
import importlib.util
import os
import subprocess
import sys

SAM3_COMMIT = "7f473cfd983d5eb43040ff608f66bff0d7bdcb53"     # facebookresearch/sam3 main, 14 Oct 2026

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if importlib.util.find_spec("sam3") is None:
    pip("--no-deps", f"git+https://github.com/facebookresearch/sam3.git@{SAM3_COMMIT}")
pip("timm>=1.0.17", "ftfy==6.1.1", "regex", "iopath", "einops", "pycocotools", "psutil")
if importlib.util.find_spec("pkg_resources") is None:          # newer setuptools dropped it; SAM 3 uses it
    pip("setuptools<81")

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

token = hf_token()
if not token:
    raise RuntimeError("Add the HF_TOKEN secret: facebook/sam3 is gated and needs an approved account.")
os.environ["HF_TOKEN"] = token        # hf_hub_download inside SAM 3 reads it from here

import torch
print(f"✓ SAM 3 installed (commit {SAM3_COMMIT[:7]}); torch {torch.__version__}, python {sys.version.split()[0]}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this notebook needs a GPU)'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 591.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 2.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sam3 0.1.0 requires numpy<2,>=1.26, but you have numpy 2.1.3 which is incompatible.


✓ SAM 3 installed (commit 7f473cf); torch 2.11.0+cu128, python 3.13.15
  GPU: Tesla T4


In [2]:
# ── Cell 2: Configuration, and the maps of a stopped earlier version ──────
import glob
import json
import re
import shutil
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/step4")
MAP_DIR = OUT_DIR / "masks" / "pre_grid2_train"     # step 5 finds this folder by its name

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
SMOKE_TEST = False                  # True = 8 training images
PROMPT = "building"
CONFIDENCE = 0.5                    # E3's value, SAM 3's own default score threshold
GRID = 2                            # E3's pre_grid2: four 512 px quarters of the before-image
MIN_INSTANCE_PX = 10                # E3's value: ignore specks
PRECISION = "fp16"                  # E3's smoke test chose fp16 on a T4 (it matched fp32 and was the fastest)
TIME_LIMIT_H = 11.0                 # stop cleanly before Kaggle's 12-hour limit
DEVICE = "cuda"

MAP_DIR.mkdir(parents=True, exist_ok=True)
previous = [Path(p) for p in sorted(glob.glob("/kaggle/input/**/pre_grid2_train", recursive=True))]
for p in previous:                  # resume: the maps a stopped earlier version wrote
    shutil.copytree(p, MAP_DIR, dirs_exist_ok=True)
print(f"✓ R16c step 4{'  [SMOKE TEST]' if SMOKE_TEST else ''}: SAM 3 '{PROMPT}' on the training before-images, "
      f"grid {GRID}, {PRECISION}; {len(list(MAP_DIR.glob('*.png'))):,} maps already here"
      + (f" (copied from {previous[0]})" if previous else ""))

✓ R16c step 4: SAM 3 'building' on the training before-images, grid 2, fp16; 0 maps already here


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: R16's training images (the split of every D4 run) and their before-images ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:
    train_pairs = train_pairs[:8]

# Cell 3 keeps only the after-image path; the before-image comes from the same manifest entries (as in E3)
pre_by_name = {Path(e["post_image_path"].replace("\\", "/")).name: e.get("pre_image_path") for e in bda_entries}
for p in train_pairs:
    p["pre_path"] = resolve_path(pre_by_name.get(Path(p["image_path"]).name))
todo = [p for p in train_pairs if not (MAP_DIR / f"{Path(p['image_path']).stem}.png").exists()]
print(f"✓ Training images: {len(train_pairs):,} (expected 5,798 when SMOKE_TEST is False); before-images found: "
      f"{sum(p['pre_path'] is not None for p in train_pairs):,}; still to do: {len(todo):,}")

✓ Training images: 5,798 (expected 5,798 when SMOKE_TEST is False); before-images found: 5,798; still to do: 5,798


In [5]:
# ── Cell 5: Load SAM 3 (with E3's bfloat16 fix) and E3's segment() ─────────
import pandas as pd
import torch.nn.functional as F
from PIL import Image
import sam3.model.vitdet as vitdet
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_sam3_image_model

def _addmm_act_any_precision(activation, linear, mat1):
    """Replaces SAM 3's fused ViT-MLP op, which always computes in bfloat16 (no bf16 tensor cores on a T4).
    Same maths, activation(linear(x)), in whatever precision the surrounding autocast uses."""
    y = F.linear(mat1, linear.weight, linear.bias)
    return F.gelu(y) if activation in (F.gelu, torch.nn.GELU) else F.relu(y)

vitdet.addmm_act = _addmm_act_any_precision        # vitdet.Mlp looks the name up at call time

t0 = time.time()
model = build_sam3_image_model(device=DEVICE)       # downloads facebook/sam3 (gated) with HF_TOKEN
processor = Sam3Processor(model, device=DEVICE, confidence_threshold=CONFIDENCE)
load_s = time.time() - t0
print(f"✓ SAM 3 loaded in {load_s:.0f} s ({sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters)")

PRECISIONS = {"fp32": nullcontext,
              "fp16": lambda: torch.autocast("cuda", dtype=torch.float16)}

def segment(image, grid=1, precision="fp32"):
    """Text prompt on an RGB uint8 image -> 16-bit instance-id map (0 = background).
    grid 2 runs the four quarters separately, so each can return up to 200 objects."""
    h, w = image.shape[:2]
    ids = np.zeros((h, w), np.uint16)
    next_id = 1
    for gy in range(grid):
        for gx in range(grid):
            y0, y1, x0, x1 = gy * h // grid, (gy + 1) * h // grid, gx * w // grid, (gx + 1) * w // grid
            with PRECISIONS[precision]():
                state = processor.set_image(Image.fromarray(image[y0:y1, x0:x1]))
                out = processor.set_text_prompt(prompt=PROMPT, state=state)
            masks = out["masks"][:, 0].cpu().numpy()
            scores = out["scores"].float().cpu().numpy()
            tile = ids[y0:y1, x0:x1]
            for k in np.argsort(scores):          # lowest score first, so higher scores win any overlap
                if masks[k].sum() >= MIN_INSTANCE_PX:
                    tile[masks[k]] = next_id
                    next_id += 1
    # ponytail: a building cut by a quarter border counts twice at grid 2, as in E3's validation maps that R16 uses
    return ids

def n_instances(ids):
    return len(np.unique(ids)) - 1                 # ids painted over by a better one no longer count

/usr/local/lib/python3.13/dist-packages/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/sam3/model/model_misc.py:71: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

sam3.pt: reconstructing file:   0%|          |  0.00B / 3.45GB            

sam3.pt: downloading bytes:           |  0.00B            

✓ SAM 3 loaded in 37 s (841 M parameters)


In [6]:
# ── Cell 6: SAM 3 on every training before-image, each map written as it comes ──
from tqdm.auto import tqdm

CLASSES = ((1, "intact"), (2, "damaged"), (3, "destroyed"))

def building_blobs(mask):
    return cv2.connectedComponents((mask > 0).astype(np.uint8), connectivity=8)[0] - 1

def score_row(found, gt, instances):
    """found: bool building mask; gt: 4-class mask of the same size (E3's per-image numbers)."""
    return {"inter": int((found & (gt > 0)).sum()), "union": int((found | (gt > 0)).sum()),
            "instances": instances, "gt_blobs": building_blobs(gt),
            **{f"gt_{c}_px": int((gt == i).sum()) for i, c in CLASSES},
            **{f"found_{c}_px": int((found & (gt == i)).sum()) for i, c in CLASSES}}

rows, stopped, t_start = [], False, time.time()
for pair in tqdm(todo, desc="Training images"):
    if (time.time() - t_start) / 3600 > TIME_LIMIT_H:
        stopped = True
        break
    image = cv2.imread(pair["pre_path"]) if pair["pre_path"] else None
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if image is None or gt is None:
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    if gt.shape != image.shape[:2]:              # same guard as E3
        gt = cv2.resize(gt, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
    t = time.time()
    ids = segment(image, GRID, PRECISION)
    stem = Path(pair["image_path"]).stem
    cv2.imwrite(str(MAP_DIR / f"{stem}.png"), ids)
    rows.append({"image": f"{stem}.png", "sec": round(time.time() - t, 2), **score_row(ids > 0, gt, n_instances(ids))})
run_hours = (time.time() - t_start) / 3600
written = len(list(MAP_DIR.glob("*.png")))
print(f"{'⚠ Stopped at the time limit' if stopped else '✓ All training images done'} after {run_hours:.1f} h; "
      f"{written:,} of {len(train_pairs):,} maps in {MAP_DIR}")

Training images:   0%|          | 0/5798 [00:00<?, ?it/s]

✓ All training images done after 4.6 h; 5,798 of 5,798 maps in /kaggle/working/step4/masks/pre_grid2_train


In [7]:
# ── Cell 7: Summary against E3's validation numbers, and the output files ──
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

per_image = pd.DataFrame(rows)
per_image.to_csv(OUT_DIR / "sam3_train.csv", index=False)     # the images this session ran
summary = pd.DataFrame([{
    "images": len(per_image),
    "building_iou": round(per_image["inter"].sum() / max(per_image["union"].sum(), 1), 4),
    "count_ratio": round(per_image["instances"].sum() / max(per_image["gt_blobs"].sum(), 1), 3),
    **{f"recall_{c}": round(per_image[f"found_{c}_px"].sum() / max(per_image[f"gt_{c}_px"].sum(), 1), 4)
       for _, c in CLASSES},
    "sec_per_image": round(float(per_image["sec"].mean()), 2)}]) if len(per_image) else pd.DataFrame()
summary.to_csv(OUT_DIR / "summary.csv", index=False)
E3_VAL = "building IoU 0.6304, count ratio 1.065, destroyed footprint recall 0.6339, 2.11 s per image"
print(summary.to_string(index=False))
print(f"E3 on the 645 validation images (pre_grid2): {E3_VAL}")

complete = written >= len(train_pairs)
config = {"task": "R16c step 4", "smoke_test": SMOKE_TEST, "sam3_commit": SAM3_COMMIT, "prompt": PROMPT,
          "confidence": CONFIDENCE, "grid": GRID, "min_instance_px": MIN_INSTANCE_PX, "precision": PRECISION,
          "model_load_s": round(load_s), "train_images": len(train_pairs), "maps": written, "complete": complete,
          "stopped_at_time_limit": stopped, "run_hours": round(run_hours, 2), "resumed_from": [str(p) for p in previous]}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# R16c step 4: SAM 3 on the training images", "",
    f"{written:,} of {len(train_pairs):,} training before-images{' (SMOKE TEST)' if SMOKE_TEST else ''}, prompt "
    f"'{PROMPT}', confidence {CONFIDENCE}, 2×2 grid, {PRECISION}, SAM 3 commit {SAM3_COMMIT[:7]}. "
    + ("Complete." if complete else "Not complete yet: attach this output and run again to continue."), "",
    "## This session's images", "", as_markdown(summary), "",
    f"E3 on the 645 validation images (pre_grid2): {E3_VAL}.", "",
    "## Use", "",
    "Attach this output to `r16c-step5-grader-on-sam3-outlines.ipynb` (step 5). "
    "It finds `masks/pre_grid2_train/` by its name.", "",
    "## Files", "",
    "- `masks/pre_grid2_train/<name>.png`: 16-bit instance maps of the before-images (0 = background), named "
    "after the after-images",
    "- `sam3_train.csv` (this session's images), `summary.csv`, `config.json`",
]), encoding="utf-8")
print(f"✓ Wrote summary.csv, sam3_train.csv, config.json and README.md to {OUT_DIR}"
      + ("" if complete else " (not complete: attach this output and run again)"))

 images  building_iou  count_ratio  recall_intact  recall_damaged  recall_destroyed  sec_per_image
   5798         0.629        1.056         0.6844          0.6703            0.6932           2.62
E3 on the 645 validation images (pre_grid2): building IoU 0.6304, count ratio 1.065, destroyed footprint recall 0.6339, 2.11 s per image
✓ Wrote summary.csv, sam3_train.csv, config.json and README.md to /kaggle/working/step4
